# Exploración de los datos

Cuaderno de trabajo para ver los tres conjuntos antes de graficar y decidir el ancho de banda del horizon graph. Parte de los CSV de `data/processed/` que genera `src/descargar_datos.py`.

In [1]:
import altair as alt
import pandas as pd

DATOS = "../data/processed/"
# el horizon tiene ~9.000 filas, más que el límite por defecto de Altair
alt.data_transformers.disable_max_rows()

DataTransformerRegistry.enable('default')

## 1. Demanda eléctrica diaria (2025)

In [2]:
demanda = pd.read_csv(DATOS + "demanda_diaria_2025.csv", parse_dates=["fecha"])
demanda.head()

,fecha,demanda_gwh
0,2025-01-01,585.942980
1,2025-01-02,714.746165
2,2025-01-03,743.131902
3,2025-01-04,672.654205
4,2025-01-05,637.473016


In [3]:
demanda["demanda_gwh"].describe()

count    365.000000
mean     702.073263
std       71.453879
min      403.297490
25%      659.396043
50%      703.280779
75%      757.495856
max      861.178961
Name: demanda_gwh, dtype: float64

In [ ]:
alt.Chart(demanda).mark_line().encode(x="fecha:T", y="demanda_gwh:Q").properties(
    width=700, height=200
)

alt.Chart(...)

Se ve el patrón semanal (los dientes de sierra), el valle de agosto y la caída del 28/04 (el "APAGÓN").
Miro la media por día de la semana:

In [ ]:
dias = ["lunes", "martes", "miércoles", "jueves", "viernes", "sábado", "domingo"]
demanda.groupby(demanda["fecha"].dt.weekday)["demanda_gwh"].mean().set_axis(dias).round(
    1
)

lunes        712.8
martes       735.9
miércoles    734.4
jueves       733.2
viernes      728.2
sábado       653.6
domingo      615.8
Name: demanda_gwh, dtype: float64

In [6]:
pd.concat([demanda.nsmallest(5, "demanda_gwh"), demanda.nlargest(5, "demanda_gwh")])

,fecha,demanda_gwh
117,2025-04-28,403.297490
123,2025-05-04,533.515623
137,2025-05-18,535.313366
116,2025-04-27,541.841685
144,2025-05-25,541.965112
182,2025-07-02,861.178961
181,2025-07-01,857.059050
183,2025-07-03,856.810160
184,2025-07-04,841.678891
197,2025-07-17,839.518237


## 2. Renta per cápita de mujeres frente a hombres

In [7]:
renta = pd.read_csv(DATOS + "renta_sexo.csv")
renta.head()

,entity,year,gni_pc__sex_male,gni_pc__sex_female,ratio
0,European Union (27),1990,44899.484,22109.568,49.242365
1,European Union (27),1991,44622.992,22483.197,50.384781
2,European Union (27),1992,44906.035,22745.840,50.652078
3,European Union (27),1993,44530.297,22752.332,51.094050
4,European Union (27),1994,45386.050,23465.008,51.700926


In [8]:
renta.pivot(index="year", columns="entity", values="ratio").round(1).tail(12)

entity,European Union (27),Spain,World
year,,,
2012,58.4,62.8,54.6
2013,59.1,63.9,54.9
2014,59.7,63.3,54.9
2015,59.8,64.6,55.0
2016,60.9,67.3,55.0
2017,61.7,67.0,55.0
2018,61.5,66.3,54.9
2019,62.3,66.0,55.6
2020,62.4,69.3,56.1


In [ ]:
alt.Chart(renta).mark_line().encode(
    x="year:O", y=alt.Y("ratio:Q", scale=alt.Scale(domain=[0, 100])), color="entity:N"
).properties(width=600, height=250)

alt.Chart(...)

España sube de forma casi continua desde 1990 (39 %) y en los últimos años se estanca alrededor del 69-71 %. Para el *delta* del gauge uso 2013 (diez años antes de 2023).

## 3. Anomalías de temperatura y ancho de banda

In [10]:
temperatura = pd.read_csv(DATOS + "anomalias_europa.csv", parse_dates=["fecha"])
temperatura.head()

,pais,fecha,anomalia
0,Noruega,1980-01-01,-4.564536
1,Noruega,1980-02-01,-3.658123
2,Noruega,1980-03-01,-2.456675
3,Noruega,1980-04-01,0.034480
4,Noruega,1980-05-01,-0.137535


In [11]:
temperatura["anomalia"].describe()

count    8976.000000
mean       -0.085298
std         1.682424
min       -12.001055
25%        -1.010767
50%        -0.035540
75%         0.966758
max         7.602699
Name: anomalia, dtype: float64

In [12]:
temperatura["anomalia"].abs().quantile([0.5, 0.75, 0.9, 0.95, 0.99]).round(2)

0.50    0.99
0.75    1.75
0.90    2.68
0.95    3.33
0.99    5.14
Name: anomalia, dtype: float64

In [13]:
alt.Chart(temperatura).mark_bar().encode(
    x=alt.X("anomalia:Q", bin=alt.Bin(step=0.5)), y="count()"
).properties(width=500, height=200)

alt.Chart(...)

Con 2 bandas, lo que pase de 2 × ancho satura. Comparo algunos anchos redondos: qué parte de los meses se queda en la primera banda y qué parte satura.

In [ ]:
v = temperatura["anomalia"].abs()
pd.DataFrame(
    {
        ancho: {
            "solo banda 1 (%)": (v <= ancho).mean() * 100,
            "satura (%)": (v > 2 * ancho).mean() * 100,
        }
        for ancho in [1, 1.5, 2]
    }
).round(1)

,1.0,1.5,2.0
solo banda 1 (%),50.5,67.9,79.9
satura (%),20.1,7.2,2.5


**Decisión: ancho = 1,5 °C** (bandas de 0-1,5 y 1,5-3 °C). Con 1 °C satura el 20 % de los meses y la banda oscura aparece en todas partes. Con 2 °C el 80 % cae en la banda clara y se pierde contraste. Con 1,5 °C, dos de cada tres meses quedan en la banda clara, la oscura se reserva para los meses realmente anómalos y solo satura un 7 % (por encima de 3 °C), sobre todo en Finlandia, Suecia y Polonia:

In [15]:
temperatura[temperatura["anomalia"].abs() > 3]["pais"].value_counts().head(5)

pais
Finlandia    115
Suecia        80
Polonia       75
Noruega       67
Austria       55
Name: count, dtype: int64

In [ ]:
espana = temperatura[temperatura["pais"] == "España"]
alt.Chart(espana).mark_area().encode(x="fecha:T", y="anomalia:Q").properties(
    width=700, height=150
)

alt.Chart(...)

In [ ]:
temperatura["decada"] = temperatura["fecha"].dt.year // 10 * 10
(temperatura.groupby("decada")["anomalia"].apply(lambda s: (s > 0).mean() * 100)).round(
    1
)

decada
1980    28.9
1990    38.6
2000    52.4
2010    58.3
2020    74.6
Name: anomalia, dtype: float64